In [ ]:
import pandas as pd
import numpy as np
import sklearn

Environment working


In [5]:
columns = [
'duration','protocol_type','service','flag','src_bytes','dst_bytes',
'land','wrong_fragment','urgent','hot','num_failed_logins',
'logged_in','num_compromised','root_shell','su_attempted',
'num_root','num_file_creations','num_shells','num_access_files',
'num_outbound_cmds','is_host_login','is_guest_login',
'count','srv_count','serror_rate','srv_serror_rate','rerror_rate',
'srv_rerror_rate','same_srv_rate','diff_srv_rate',
'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
'dst_host_same_srv_rate','dst_host_diff_srv_rate',
'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
'dst_host_serror_rate','dst_host_srv_serror_rate',
'dst_host_rerror_rate','dst_host_srv_rerror_rate','label'
]

df = pd.read_csv("../data/KDDTrain+.txt", names=columns)

In [6]:
df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label
0,tcp,ftp_data,SF,491,0,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
0,udp,other,SF,146,0,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
0,tcp,private,S0,0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
0,tcp,http,SF,232,8153,0,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
0,tcp,http,SF,199,420,0,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [7]:
df.shape

(125973, 42)

In [8]:
columns = [
'duration','protocol_type','service','flag','src_bytes','dst_bytes',
'land','wrong_fragment','urgent','hot','num_failed_logins',
'logged_in','num_compromised','root_shell','su_attempted',
'num_root','num_file_creations','num_shells','num_access_files',
'num_outbound_cmds','is_host_login','is_guest_login',
'count','srv_count','serror_rate','srv_serror_rate','rerror_rate',
'srv_rerror_rate','same_srv_rate','diff_srv_rate',
'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
'dst_host_same_srv_rate','dst_host_diff_srv_rate',
'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
'dst_host_serror_rate','dst_host_srv_serror_rate',
'dst_host_rerror_rate','dst_host_srv_rerror_rate',
'label','difficulty'
]
df = pd.read_csv("../data/KDDTrain+.txt", names=columns)
df['label'].value_counts()

label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64

In [10]:
attack_mapping = {
'normal':'normal',

'neptune':'dos',
'smurf':'dos',
'back':'dos',
'teardrop':'dos',
'pod':'dos',
'land':'dos',

'satan':'probe',
'ipsweep':'probe',
'portsweep':'probe',
'nmap':'probe',

'guess_passwd':'r2l',
'ftp_write':'r2l',
'imap':'r2l',
'phf':'r2l',
'multihop':'r2l',
'warezmaster':'r2l',
'warezclient':'r2l',
'spy':'r2l',

'buffer_overflow':'u2r',
'rootkit':'u2r',
'loadmodule':'u2r',
'perl':'u2r'
}

df['attack_class'] = df['label'].map(attack_mapping)
df['attack_class'].value_counts()

attack_class
normal    67343
dos       45927
probe     11656
r2l         995
u2r          52
Name: count, dtype: int64

In [15]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib

# Target (5 classes)
y = df['attack_class']

# Features
X = df.drop(['label','attack_class'], axis=1)

# Encode categorical columns
cat_cols = X.select_dtypes(include=['object']).columns

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# Normalize numeric features
numeric_cols = X.select_dtypes(include=['int64','float64']).columns

feature_names = X.columns.tolist()
joblib.dump(feature_names, '../models/feature_names.pkl')
scaler = MinMaxScaler()

X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

print("Feature count after scaling:", X.shape)

from sklearn.preprocessing import MinMaxScaler

# Refit scaler on final feature set
scaler = MinMaxScaler()

scaler.fit(X)

print("Scaler expects:", scaler.n_features_in_)

# Train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Handle class imbalance
smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
train_columns = X_train.columns
train_columns.to_series().to_csv("../data/train_columns.csv", index=False)
print("SMOTE Applied")
print("Training shape:", X_train_res.shape)

print("Class distribution after SMOTE:\n")
print(y_train_res.value_counts())

Feature count after scaling: (125973, 42)
Scaler expects: 42
SMOTE Applied
Training shape: (269370, 42)
Class distribution after SMOTE:

attack_class
dos       53874
normal    53874
probe     53874
r2l       53874
u2r       53874
Name: count, dtype: int64


In [16]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Fit scaler on final feature matrix
scaler.fit(X)

print("Scaler expects:", scaler.n_features_in_)

Scaler expects: 42


In [17]:
# Train Model

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train_res, y_train_res)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [18]:
print("Training feature count:", X_train_res.shape[1])
print("Validation feature count:", X_val.shape[1])
print("Model expects:", rf_model.n_features_in_)
print("Scaler expects:", scaler.n_features_in_)

Training feature count: 42
Validation feature count: 42
Model expects: 42
Scaler expects: 42


In [19]:
# Prediction

y_pred = rf_model.predict(X_val)

In [20]:
# Evaluation

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Accuracy:", accuracy_score(y_val, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_val, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_val, y_pred))

Accuracy: 0.9992458821194682

Classification Report:

              precision    recall  f1-score   support

         dos       1.00      1.00      1.00      9186
      normal       1.00      1.00      1.00     13469
       probe       1.00      1.00      1.00      2331
         r2l       0.98      0.98      0.98       199
         u2r       1.00      0.90      0.95        10

    accuracy                           1.00     25195
   macro avg       1.00      0.98      0.98     25195
weighted avg       1.00      1.00      1.00     25195


Confusion Matrix:

[[ 9185     0     1     0     0]
 [    1 13460     5     3     0]
 [    0     3  2327     1     0]
 [    0     4     0   195     0]
 [    0     1     0     0     9]]


In [63]:
print("Training feature count:", X_train_res.shape[1])
print("Validation feature count:", X_val.shape[1])
print("Model expects:", rf_model.n_features_in_)
print("Scaler expects:", scaler.n_features_in_)

Training feature count: 124
Validation feature count: 124
Model expects: 124
Scaler expects: 124


In [21]:


joblib.dump(rf_model, "../models/threat_detection_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")

print("Saved new model and scaler")

Saved new model and scaler


In [ ]:
# Preprocessing Test data
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

columns = [
'duration','protocol_type','service','flag','src_bytes','dst_bytes',
'land','wrong_fragment','urgent','hot','num_failed_logins',
'logged_in','num_compromised','root_shell','su_attempted',
'num_root','num_file_creations','num_shells','num_access_files',
'num_outbound_cmds','is_host_login','is_guest_login',
'count','srv_count','serror_rate','srv_serror_rate','rerror_rate',
'srv_rerror_rate','same_srv_rate','diff_srv_rate',
'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
'dst_host_same_srv_rate','dst_host_diff_srv_rate',
'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
'dst_host_serror_rate','dst_host_srv_serror_rate',
'dst_host_rerror_rate','dst_host_srv_rerror_rate','label'
]

df = pd.read_csv("../data/KDDTest+.txt", names=columns)

y = df['label']
X = df.drop(['label'], axis=1)
X = pd.get_dummies(X)
print("Feature count after encoding:", X.shape)

scaler = MinMaxScaler()
X = scaler.fit_transform(X)

X = pd.DataFrame(X)

print("Final shape:", X.shape)

X.to_csv("../data/network_test_data.csv", index=False)

Feature count after encoding: (22544, 153)
Final shape: (22544, 153)


In [2]:

import joblib
import os

# Save the list of 153 column names
if not os.path.exists('models'):
    os.makedirs('models')

feature_names = X.columns.tolist()
joblib.dump(feature_names, 'models/feature_names.pkl')

['models/feature_names.pkl']